# Data Cleaning

**Purpose:** Clean and normalize the raw extracted text. Remove noise, fix encoding, and prepare clean pages for chunking.

**Pipeline Position:** `data/extracted/*.json` → **[DATA CLEANING]** → `data/cleaned/*.json`

---

### What this notebook does:
1. Loads extracted JSON files from `data/extracted/`
2. Applies a text cleaning pipeline:
   - Removes excessive whitespace, newlines, and special characters
   - Fixes hyphenation and line break artifacts from PDF extraction
   - Removes headers/footers (page numbers, repeated boilerplate)
   - Filters out pages with insufficient content (< min character threshold)
3. Saves cleaned output to `data/cleaned/`

### Input from:
`data/extracted/<filename>.json`

### Output saved to:
`data/cleaned/<filename>.json`

---

In [1]:
import json
import re
import os
from pathlib import Path

print("Libraries imported")

Libraries imported


In [ ]:
EXTRACTED_DIR = Path("../../data/extracted")
CLEANED_DIR = Path("../../data/cleaned")

CLEANED_DIR.mkdir(parents=True, exist_ok=True)

# Minimum characters required to keep a page (skip near-empty pages)
MIN_CHAR_THRESHOLD = 80

extracted_files = sorted(EXTRACTED_DIR.glob("*.json"))
print(f"Found {len(extracted_files)} extracted files")
for f in extracted_files:
    print(f"  - {f.name}")

Found 16 extracted files
  - ABL-Business-User-Guidelines.json
  - ABL-FAQs.json
  - Bank-Alfalah-FAQs-Account-Opening.json
  - Bank-Alfalah-FAQs-Personal-Loan.json
  - Bank-Alfalah-Self-Service-Banking.json
  - HBL-FAQs-Home-Remittance.json
  - HBL-FAQs.json
  - HBL-Islamic-Current-Account.json
  - HBL-Work-Conventional-Accounts.json
  - Meezan-Bank-FAQs-Debit-Cards.json
  - Meezan-Bank-FAQs-Digital-Account.json
  - Meezan-Bank-FAQs-Roshan-Apna-Ghar.json
  - Meezan-Bank-FAQs-Roshan-Digital-Account.json
  - Meezan-Bank-FAQs-Salaried.json
  - State-Bank-FAQs-History.json
  - State-Bank-FAQs.json


In [4]:
# Word rejoining function
def fix_hyphenation(text: str) -> str:
    """Rejoin words broken across lines with a hyphen (PDF artifact)."""
    # E.g., 'remit-\ntance' → 'remittance'
    return re.sub(r'(\w+)-\n(\w+)', r'\1\2', text)


# Whitespace normalization function
def normalize_whitespace(text: str) -> str:
    """Collapse multiple spaces/tabs into one, strip leading/trailing whitespace."""
    text = re.sub(r'[ \t]+', ' ', text)   # multiple spaces/tabs → single space
    text = re.sub(r'\n{3,}', '\n\n', text)  # 3+ newlines → max 2
    return text.strip()


# Page number removal function
def remove_page_numbers(text: str) -> str:
    """Remove standalone page number lines like 'Page 1 of 10' or just '1'."""
    text = re.sub(r'(?im)^\s*page\s+\d+\s*(of\s+\d+)?\s*$', '', text)
    text = re.sub(r'(?m)^\s*\d+\s*$', '', text)  # lone numbers on a line
    return text


# Special character removal function
def remove_special_chars(text: str) -> str:
    """Remove non-printable and unwanted unicode control characters."""
    # Keep printable ASCII + common Urdu/Arabic unicode range if present
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)
    return text


# Full cleaning pipeline function
def clean_text(raw_text: str) -> str:
    """Full cleaning pipeline applied to a single page's text."""
    text = fix_hyphenation(raw_text)
    text = remove_special_chars(text)
    text = remove_page_numbers(text)
    text = normalize_whitespace(text)
    return text


print("Cleaning functions defined")

Cleaning functions defined


In [5]:
total_pages_in = 0
total_pages_out = 0

for json_file in extracted_files:
    with open(json_file, "r", encoding="utf-8") as f:
        pages = json.load(f)

    cleaned_pages = []
    skipped = 0

    for page in pages:
        cleaned = clean_text(page["raw_text"])

        # Skip near-empty pages
        if len(cleaned) < MIN_CHAR_THRESHOLD:
            skipped += 1
            continue

        # Build cleaned page dict (keep original metadata, add cleaned_text)
        cleaned_pages.append({
            "bank_name":   page["bank_name"],
            "source_file": page["source_file"],
            "page_number": page["page_number"],
            "total_pages": page["total_pages"],
            "cleaned_text": cleaned
        })

    total_pages_in += len(pages)
    total_pages_out += len(cleaned_pages)

    # Save cleaned file
    out_path = CLEANED_DIR / json_file.name
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(cleaned_pages, f, ensure_ascii=False, indent=2)

    print(f"{json_file.stem:<45}  {len(pages):>3} pages in  |  {skipped:>2} skipped  |  {len(cleaned_pages):>3} saved")

print(f"\nTotal: {total_pages_in} pages extracted → {total_pages_out} pages kept after cleaning")

ABL-Business-User-Guidelines                     7 pages in  |   1 skipped  |    6 saved
ABL-FAQs                                        19 pages in  |   0 skipped  |   19 saved
Bank-Alfalah-FAQs-Account-Opening                8 pages in  |   0 skipped  |    8 saved
Bank-Alfalah-FAQs-Personal-Loan                  3 pages in  |   0 skipped  |    3 saved
Bank-Alfalah-Self-Service-Banking                5 pages in  |   0 skipped  |    5 saved
HBL-FAQs-Home-Remittance                         2 pages in  |   0 skipped  |    2 saved
HBL-FAQs                                         5 pages in  |   0 skipped  |    5 saved
HBL-Islamic-Current-Account                      4 pages in  |   0 skipped  |    4 saved
HBL-Work-Conventional-Accounts                   6 pages in  |   2 skipped  |    4 saved
Meezan-Bank-FAQs-Debit-Cards                     2 pages in  |   0 skipped  |    2 saved
Meezan-Bank-FAQs-Digital-Account                 3 pages in  |   0 skipped  |    3 saved
Meezan-Bank-FAQs-Rosh

In [6]:
# Pick a sample file + page to compare before/after
SAMPLE_FILE = extracted_files[0].name  # change to any filename
SAMPLE_PAGE_INDEX = 1  # 0-based index

# Load raw
with open(EXTRACTED_DIR / SAMPLE_FILE, "r", encoding="utf-8") as f:
    raw_pages = json.load(f)

# Load cleaned
with open(CLEANED_DIR / SAMPLE_FILE, "r", encoding="utf-8") as f:
    cleaned_pages = json.load(f)

# Compare
print("=" * 60)
print("🔴 BEFORE CLEANING (raw_text)")
print("=" * 60)
print(raw_pages[SAMPLE_PAGE_INDEX]["raw_text"][:600])

print("\n" + "=" * 60)
print("🟢 AFTER CLEANING (cleaned_text)")
print("=" * 60)
print(cleaned_pages[SAMPLE_PAGE_INDEX]["cleaned_text"][:600])

🔴 BEFORE CLEANING (raw_text)
 
OBDX – General Guidelines 
Table of Contents 
1. 
1.1. 
1.2. 
1.3. 
2. 
3. 
3.1. 
3.2. 
3.3. 
4. 
Page- 2 
 
 
 
myABL Verify (Soft Token App) Configuration ....................................................................... 3 
Introduction ...................................................................................................................................... 3 
Downloading/Installing myABL Verify App ....................................................................................... 3 
Login Process of BIB & Configuring/Syncing myABL Verify App for OTP ................

🟢 AFTER CLEANING (cleaned_text)
OBDX – General Guidelines 
1. myABL Verfiy (Soft Token App) Configuration 
1.1. Introduction 
myABL Verify (Soft Token App) generates one-time password (OTP) on your device for two-factor 
authentication. In addition to username and password, the one-time password generated by this app 
will be required to sign-in to the account, makin

In [ ]:
# Final summary of cleaning results
print("=" * 50)
print(f"Pages before cleaning : {total_pages_in}")
print(f"Pages after cleaning  : {total_pages_out}")
print(f"Pages filtered out    : {total_pages_in - total_pages_out}")
print(f"Retention rate        : {total_pages_out / total_pages_in * 100:.1f}%")
print(f"Output location       : {CLEANED_DIR}")
print("=" * 50)

Pages before cleaning : 114
Pages after cleaning  : 111
Pages filtered out    : 3
Retention rate        : 97.4%
Output location       : ..\data\cleaned
